# Data Architecture — Trabalho da aula 2
### Grupo 2: Samuel Cupertino, Ksenia Busquet, Henrique Arduini, Renato de Jesus Rocha


**Departamento escolhido: Eletrônicos.**


In [1]:
import sqlite3
import os
from datetime import datetime, timedelta
import pandas as pd
from IPython.display import display

DB_PATH = "laboratorio_eletronicos.db"

# Remove o banco anterior para que o notebook possa ser re-executado do zero
# o que evita erros de "table already exists" ou dados duplicados em reexecuções).
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)

conn.execute("PRAGMA foreign_keys = ON")

cursor = conn.cursor()
print("Conexão aberta em", DB_PATH, "| foreign_keys =", conn.execute("PRAGMA foreign_keys").fetchone()[0])


Conexão aberta em laboratorio_eletronicos.db | foreign_keys = 1


### Script base

In [2]:
cursor.executescript('''
CREATE TABLE departamento (
    id_departamento INTEGER PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    responsavel VARCHAR(100)
);

CREATE TABLE invent_mestre (
    id_item INTEGER PRIMARY KEY,
    codigo_sku VARCHAR(50) NOT NULL UNIQUE,
    descricao VARCHAR(200) NOT NULL,
    id_departamento INTEGER NOT NULL,
    unidade_medida VARCHAR(20) NOT NULL,
    preco_custo DECIMAL(10,2) NOT NULL,
    FOREIGN KEY (id_departamento) REFERENCES departamento(id_departamento)
);

CREATE TABLE transac_mestre (
    id_transacao INTEGER PRIMARY KEY,
    id_item INTEGER NOT NULL,
    data_transacao DATE NOT NULL,
    tipo VARCHAR(10) NOT NULL CHECK (tipo IN ('ENTRADA', 'SAIDA', 'AJUSTE')),
    quantidade INTEGER NOT NULL,
    origem_destino VARCHAR(200),
    FOREIGN KEY (id_item) REFERENCES invent_mestre(id_item)
);
''')
conn.commit()
print("Tabelas departamento, invent_mestre e transac_mestre criadas.")


Tabelas departamento, invent_mestre e transac_mestre criadas.


## Parte 1 — Modelagem do departamento com IA generativa

### Prompt utilizado

> "Preciso modelar as tabelas de um departamento de **Eletrônicos** para um sistema de armazém. A tabela `invent_mestre` já existe com os campos `id_item` (PK), `codigo_sku`, `descricao`, `id_departamento`, `unidade_medida` e `preco_custo`. Crie uma tabela específica para Eletrônicos com FK para `invent_mestre`, incluindo os atributos relevantes para esse tipo de produto (ex.: marca, garantia, voltagem, potência). Use VARCHAR com tamanho definido e INTEGER para chaves."

### Resultado gerado pela IA


In [3]:
cursor.execute('''
CREATE TABLE invent_eletronicos (
    id_item INTEGER PRIMARY KEY,
    marca VARCHAR(50) NOT NULL,
    garantia_meses INTEGER NOT NULL,
    voltagem VARCHAR(10),
    potencia_watts INTEGER,
    cor VARCHAR(30),
    FOREIGN KEY (id_item) REFERENCES invent_mestre(id_item)
)
''')
conn.commit()
print("Tabela invent_eletronicos criada.")


Tabela invent_eletronicos criada.


### Avaliação do resultado

* **A FK está corretamente referenciando `invent_mestre.id_item`?** Sim. `id_item` é ao mesmo tempo PK de `invent_eletronicos` e FK para `invent_mestre.id_item` — a tabela funciona como uma **extensão 1:1** do item mestre (cada item de eletrônico tem exatamente um registro de atributos específicos).
* **Os atributos fazem sentido para o tipo de produto?** Sim — `marca`, `garantia_meses`, `voltagem`, `potencia_watts` e `cor` são atributos típicos de um item eletrônico e não fariam sentido genérico em `invent_mestre` (que é compartilhada por todos os departamentos).
* **Os tipos de dados são adequados?** Sim: `VARCHAR` com tamanho definido, `INTEGER` para chaves e para campos numéricos discretos (garantia em meses, potência em watts).
* **Há atributos importantes que a IA deixou de fora?** Sim — pedimos à IA uma segunda rodada e ela sugeriu: `peso_kg`, `dimensoes` (cm), `certificacao_anatel` (para produtos com wi-fi) e `consumo_kwh_mes`. Não os incluímos na tabela final para manter o exercício enxuto, mas ficam registrados aqui como próxima iteração do modelo.


### Passo final — INSERTs gerados (departamento, itens, atributos e movimentações)

As datas de `transac_mestre` são geradas **relativas ao momento de execução** do notebook (não fixas), para que a pergunta "últimos 30 dias" (Parte 2) funcione corretamente sempre que o notebook for rodado.

In [4]:
cursor.execute(
    "INSERT INTO departamento (id_departamento, nome, responsavel) VALUES (?, ?, ?)",
    (1, 'Eletrônicos', 'Bruno Busquet')
)

itens = [
    (1,  'SKU-ELE-001', 'Notebook Pro 15',           1, 'UN', 3500.00),
    (2,  'SKU-ELE-002', 'Monitor UltraWide 29"',     1, 'UN', 1200.00),
    (3,  'SKU-ELE-003', 'Teclado Mecânico RGB',      1, 'UN',  250.00),
    (4,  'SKU-ELE-004', 'Mouse Gamer Óptico',        1, 'UN',  150.00),
    (5,  'SKU-ELE-005', 'Fone Bluetooth ANC',        1, 'UN',  600.00),
    (6,  'SKU-ELE-006', 'Webcam Full HD',            1, 'UN',  220.00),
    (7,  'SKU-ELE-007', 'SSD NVMe 1TB',              1, 'UN',  450.00),
    (8,  'SKU-ELE-008', 'Roteador Wi-Fi 6',          1, 'UN',  380.00),
    (9,  'SKU-ELE-009', 'Carregador USB-C 65W',      1, 'UN',  120.00),
    (10, 'SKU-ELE-010', 'Impressora Multifuncional', 1, 'UN',  900.00),
]
cursor.executemany(
    "INSERT INTO invent_mestre (id_item, codigo_sku, descricao, id_departamento, unidade_medida, preco_custo) VALUES (?,?,?,?,?,?)",
    itens
)

atributos = [
    (1,  'TechPro',    12, 'Bivolt', 65, 'Prata'),
    (2,  'VisionMax',  24, 'Bivolt', 45, 'Preto'),
    (3,  'KeyForce',   12, 'Bivolt',  5, 'Preto'),
    (4,  'KeyForce',   12, 'Bivolt',  3, 'Preto'),
    (5,  'SoundWave',  12, None,      2, 'Branco'),
    (6,  'ClearView',   6, 'Bivolt',  3, 'Preto'),
    (7,  'DataFast',   60, None,      4, None),
    (8,  'NetSpeed',   24, 'Bivolt', 12, 'Preto'),
    (9,  'PowerGo',     6, 'Bivolt', 65, 'Branco'),
    (10, 'PrintAll',   12, 'Bivolt', 30, 'Branco'),
]
cursor.executemany(
    "INSERT INTO invent_eletronicos (id_item, marca, garantia_meses, voltagem, potencia_watts, cor) VALUES (?,?,?,?,?,?)",
    atributos
)

hoje = datetime.now()
def dias_atras(n):
    return (hoje - timedelta(days=n)).strftime('%Y-%m-%d')

transacoes = [
    (1,  1, dias_atras(75), 'ENTRADA', 30,  'Fornecedor TechPro'),
    (2,  2, dias_atras(75), 'ENTRADA', 40,  'Fornecedor VisionMax'),
    (3,  3, dias_atras(74), 'ENTRADA', 60,  'Fornecedor KeyForce'),
    (4,  4, dias_atras(74), 'ENTRADA', 80,  'Fornecedor KeyForce'),
    (5,  5, dias_atras(73), 'ENTRADA', 50,  'Fornecedor SoundWave'),
    (6,  6, dias_atras(73), 'ENTRADA', 45,  'Fornecedor ClearView'),
    (7,  7, dias_atras(72), 'ENTRADA', 70,  'Fornecedor DataFast'),
    (8,  8, dias_atras(72), 'ENTRADA', 35,  'Fornecedor NetSpeed'),
    (9,  9, dias_atras(71), 'ENTRADA', 100, 'Fornecedor PowerGo'),
    (10, 10, dias_atras(71), 'ENTRADA', 20, 'Fornecedor PrintAll'),
    (11, 1, dias_atras(40), 'SAIDA', 5,   'Loja Centro'),
    (12, 2, dias_atras(35), 'SAIDA', 8,   'Loja Centro'),
    (13, 1, dias_atras(10), 'SAIDA', 4,   'Loja Shopping'),
    (14, 3, dias_atras(12), 'SAIDA', 15,  'Loja Shopping'),
    (15, 4, dias_atras(9),  'SAIDA', 20,  'E-commerce'),
    (16, 5, dias_atras(8),  'SAIDA', 10,  'E-commerce'),
    (17, 2, dias_atras(5),  'SAIDA', 6,   'Loja Centro'),
    (18, 7, dias_atras(42), 'SAIDA', 25,  'Loja Shopping'),
    (19, 9, dias_atras(4),  'SAIDA', 40,  'E-commerce'),
    (20, 10, dias_atras(2), 'SAIDA', 5,   'Loja Centro'),
    (21, 3, dias_atras(11), 'AJUSTE', -2, 'Inventário físico - quebra'),
    (22, 6, dias_atras(10), 'AJUSTE', 3,  'Inventário físico - sobra'),
]
cursor.executemany(
    "INSERT INTO transac_mestre (id_transacao, id_item, data_transacao, tipo, quantidade, origem_destino) VALUES (?,?,?,?,?,?)",
    transacoes
)
conn.commit()

print("Registros inseridos:")
print(" - departamento:", cursor.execute("SELECT COUNT(*) FROM departamento").fetchone()[0])
print(" - invent_mestre:", cursor.execute("SELECT COUNT(*) FROM invent_mestre").fetchone()[0])
print(" - invent_eletronicos:", cursor.execute("SELECT COUNT(*) FROM invent_eletronicos").fetchone()[0])
print(" - transac_mestre:", cursor.execute("SELECT COUNT(*) FROM transac_mestre").fetchone()[0])

display(pd.read_sql_query("SELECT * FROM invent_mestre", conn))
display(pd.read_sql_query("SELECT * FROM invent_eletronicos", conn))
display(pd.read_sql_query("SELECT * FROM transac_mestre ORDER BY id_item, data_transacao", conn))


Registros inseridos:
 - departamento: 1
 - invent_mestre: 10
 - invent_eletronicos: 10
 - transac_mestre: 22


,id_item,codigo_sku,descricao,id_departamento,unidade_medida,preco_custo
0,1,SKU-ELE-001,Notebook Pro 15,1,UN,3500
1,2,SKU-ELE-002,"Monitor UltraWide 29""",1,UN,1200
2,3,SKU-ELE-003,Teclado Mecânico RGB,1,UN,250
3,4,SKU-ELE-004,Mouse Gamer Óptico,1,UN,150
4,5,SKU-ELE-005,Fone Bluetooth ANC,1,UN,600
5,6,SKU-ELE-006,Webcam Full HD,1,UN,220
6,7,SKU-ELE-007,SSD NVMe 1TB,1,UN,450
7,8,SKU-ELE-008,Roteador Wi-Fi 6,1,UN,380
8,9,SKU-ELE-009,Carregador USB-C 65W,1,UN,120
9,10,SKU-ELE-010,Impressora Multifuncional,1,UN,900


,id_item,marca,garantia_meses,voltagem,potencia_watts,cor
0,1,TechPro,12,Bivolt,65,Prata
1,2,VisionMax,24,Bivolt,45,Preto
2,3,KeyForce,12,Bivolt,5,Preto
3,4,KeyForce,12,Bivolt,3,Preto
4,5,SoundWave,12,None,2,Branco
5,6,ClearView,6,Bivolt,3,Preto
6,7,DataFast,60,None,4,None
7,8,NetSpeed,24,Bivolt,12,Preto
8,9,PowerGo,6,Bivolt,65,Branco
9,10,PrintAll,12,Bivolt,30,Branco


,id_transacao,id_item,data_transacao,tipo,quantidade,origem_destino
0,1,1,2026-06-04,ENTRADA,30,Fornecedor TechPro
1,11,1,2026-07-09,SAIDA,5,Loja Centro
2,13,1,2026-08-08,SAIDA,4,Loja Shopping
3,2,2,2026-06-04,ENTRADA,40,Fornecedor VisionMax
4,12,2,2026-07-14,SAIDA,8,Loja Centro
5,17,2,2026-08-13,SAIDA,6,Loja Centro
6,3,3,2026-06-05,ENTRADA,60,Fornecedor KeyForce
7,14,3,2026-08-06,SAIDA,15,Loja Shopping
8,21,3,2026-08-07,AJUSTE,-2,Inventário físico - quebra
9,4,4,2026-06-05,ENTRADA,80,Fornecedor KeyForce


In [5]:
# Checagem de consistência: o saldo de cada item (ENTRADA - SAIDA +/- AJUSTE)
# nunca pode ficar negativo, senão as movimentações inseridas não são plausíveis.
query_saldo = '''
SELECT
    im.descricao,
    SUM(CASE tm.tipo
            WHEN 'ENTRADA' THEN tm.quantidade
            WHEN 'SAIDA'   THEN -tm.quantidade
            ELSE tm.quantidade  -- AJUSTE já vem com sinal (+ ou -)
        END) AS saldo_estimado
FROM invent_mestre im
JOIN transac_mestre tm ON im.id_item = tm.id_item
GROUP BY im.id_item
ORDER BY im.id_item
'''
df_saldo = pd.read_sql_query(query_saldo, conn)
display(df_saldo)
assert (df_saldo['saldo_estimado'] >= 0).all(), "Há item com saldo negativo — movimentações inconsistentes!"
print("OK: todos os saldos são >= 0, as movimentações são consistentes com o estoque.")


,descricao,saldo_estimado
0,Notebook Pro 15,21
1,"Monitor UltraWide 29""",26
2,Teclado Mecânico RGB,43
3,Mouse Gamer Óptico,60
4,Fone Bluetooth ANC,40
5,Webcam Full HD,48
6,SSD NVMe 1TB,45
7,Roteador Wi-Fi 6,35
8,Carregador USB-C 65W,60
9,Impressora Multifuncional,15


OK: todos os saldos são >= 0, as movimentações são consistentes com o estoque.


### Avaliação do resultado

* **Estão satisfeitos com o modelo?** Sim. A extensão 1:1 (`invent_eletronicos` referenciando `invent_mestre` pelo mesmo `id_item`) é simples e evita duplicar campos genéricos (SKU, descrição, preço) que já vivem no mestre.
* **Algo chamou a atenção?** O SQLite aceita `DECIMAL(10,2)` na definição da coluna, mas internamente armazena o valor com afinidade numérica (`REAL`/`NUMERIC`), não impõe de fato duas casas decimais fixas — isso é uma particularidade do SQLite (bancos como PostgreSQL/MySQL fariam isso valer de verdade). Não é um bug, mas vale ter em mente ao migrar para outro SGBD.
* **Mudariam algo no modelo mestre?** Duas coisas ficaram como sugestão de evolução: (1) `invent_mestre`/`departamento` não têm colunas de auditoria (`criado_em`, `atualizado_em`), que ajudariam a rastrear mudanças de cadastro; (2) hoje o saldo de estoque é sempre **derivado** somando `transac_mestre` a cada consulta — funciona bem para o volume deste laboratório, mas em um cenário real com milhões de transações valeria ter uma tabela de saldo materializado, recalculada por trigger ou job.


## Parte 2 — Execução, integridade e consultas

### Verificação de integridade

Tentando inserir um item com `id_departamento` inexistente (999) — como `PRAGMA foreign_keys = ON` foi habilitado no início, isso deve falhar.

In [6]:
try:
    cursor.execute(
        "INSERT INTO invent_mestre (id_item, codigo_sku, descricao, id_departamento, unidade_medida, preco_custo) VALUES (?,?,?,?,?,?)",
        (99, 'SKU-TESTE-999', 'Item de teste (departamento inválido)', 999, 'UN', 10.00)
    )
    conn.commit()
    print("Nada impediu o insert — havia um problema na FK ou na PRAGMA!")
except sqlite3.IntegrityError as erro:
    conn.rollback()
    print("Erro obtido, como esperado:")
    print(" ->", erro)


Erro obtido, como esperado:
 -> FOREIGN KEY constraint failed


### JOIN entre `invent_eletronicos` e `invent_mestre`

In [7]:
query_join = '''
SELECT
    im.codigo_sku,
    im.descricao,
    ie.marca,
    ie.voltagem,
    ie.potencia_watts,
    ie.garantia_meses,
    im.preco_custo
FROM invent_eletronicos ie
JOIN invent_mestre im ON ie.id_item = im.id_item
ORDER BY im.id_item
'''
display(pd.read_sql_query(query_join, conn))


,codigo_sku,descricao,marca,voltagem,potencia_watts,garantia_meses,preco_custo
0,SKU-ELE-001,Notebook Pro 15,TechPro,Bivolt,65,12,3500
1,SKU-ELE-002,"Monitor UltraWide 29""",VisionMax,Bivolt,45,24,1200
2,SKU-ELE-003,Teclado Mecânico RGB,KeyForce,Bivolt,5,12,250
3,SKU-ELE-004,Mouse Gamer Óptico,KeyForce,Bivolt,3,12,150
4,SKU-ELE-005,Fone Bluetooth ANC,SoundWave,None,2,12,600
5,SKU-ELE-006,Webcam Full HD,ClearView,Bivolt,3,6,220
6,SKU-ELE-007,SSD NVMe 1TB,DataFast,None,4,60,450
7,SKU-ELE-008,Roteador Wi-Fi 6,NetSpeed,Bivolt,12,24,380
8,SKU-ELE-009,Carregador USB-C 65W,PowerGo,Bivolt,65,6,120
9,SKU-ELE-010,Impressora Multifuncional,PrintAll,Bivolt,30,12,900


### Perguntas analíticas

**1. Todos os itens do departamento com descrição, SKU e preço de custo, ordenados por preço decrescente:**

In [8]:
q1 = '''
SELECT codigo_sku, descricao, preco_custo
FROM invent_mestre
WHERE id_departamento = 1
ORDER BY preco_custo DESC
'''
display(pd.read_sql_query(q1, conn))


,codigo_sku,descricao,preco_custo
0,SKU-ELE-001,Notebook Pro 15,3500
1,SKU-ELE-002,"Monitor UltraWide 29""",1200
2,SKU-ELE-010,Impressora Multifuncional,900
3,SKU-ELE-005,Fone Bluetooth ANC,600
4,SKU-ELE-007,SSD NVMe 1TB,450
5,SKU-ELE-008,Roteador Wi-Fi 6,380
6,SKU-ELE-003,Teclado Mecânico RGB,250
7,SKU-ELE-006,Webcam Full HD,220
8,SKU-ELE-004,Mouse Gamer Óptico,150
9,SKU-ELE-009,Carregador USB-C 65W,120


**2. Quantidade de movimentações por tipo (ENTRADA, SAIDA, AJUSTE) para itens do departamento:**

In [9]:
q2 = '''
SELECT tm.tipo, COUNT(*) AS qtd_movimentacoes
FROM transac_mestre tm
JOIN invent_mestre im ON tm.id_item = im.id_item
WHERE im.id_departamento = 1
GROUP BY tm.tipo
ORDER BY qtd_movimentacoes DESC
'''
display(pd.read_sql_query(q2, conn))


,tipo,qtd_movimentacoes
0,SAIDA,10
1,ENTRADA,10
2,AJUSTE,2


**3. Item do departamento com maior volume total movimentado (soma de quantidade em todas as transações):**

In [10]:
q3 = '''
SELECT im.descricao, SUM(ABS(tm.quantidade)) AS volume_total
FROM transac_mestre tm
JOIN invent_mestre im ON tm.id_item = im.id_item
WHERE im.id_departamento = 1
GROUP BY im.id_item, im.descricao
ORDER BY volume_total DESC
'''
df_q3 = pd.read_sql_query(q3, conn)
display(df_q3)
print(f"Item com maior volume: {df_q3.iloc[0]['descricao']} ({df_q3.iloc[0]['volume_total']} unidades movimentadas)")


,descricao,volume_total
0,Carregador USB-C 65W,140
1,Mouse Gamer Óptico,100
2,SSD NVMe 1TB,95
3,Teclado Mecânico RGB,77
4,Fone Bluetooth ANC,60
5,"Monitor UltraWide 29""",54
6,Webcam Full HD,48
7,Notebook Pro 15,39
8,Roteador Wi-Fi 6,35
9,Impressora Multifuncional,25


Item com maior volume: Carregador USB-C 65W (140 unidades movimentadas)


**4. Itens com pelo menos uma movimentação do tipo SAIDA nos últimos 30 dias, com a quantidade total saída por item:**

In [11]:
q4 = '''
SELECT im.descricao, SUM(tm.quantidade) AS qtd_saida_ultimos_30_dias
FROM transac_mestre tm
JOIN invent_mestre im ON tm.id_item = im.id_item
WHERE im.id_departamento = 1
  AND tm.tipo = 'SAIDA'
  AND tm.data_transacao >= date('now', '-30 day')
GROUP BY im.id_item, im.descricao
ORDER BY qtd_saida_ultimos_30_dias DESC
'''
display(pd.read_sql_query(q4, conn))


,descricao,qtd_saida_ultimos_30_dias
0,Carregador USB-C 65W,40
1,Mouse Gamer Óptico,20
2,Teclado Mecânico RGB,15
3,Fone Bluetooth ANC,10
4,"Monitor UltraWide 29""",6
5,Impressora Multifuncional,5
6,Notebook Pro 15,4


**5. JOIN entre a tabela do departamento e `invent_mestre`, mostrando os atributos específicos junto com SKU e preço de custo:**

In [12]:
q5 = '''
SELECT
    im.codigo_sku,
    im.descricao,
    ie.marca,
    ie.garantia_meses,
    ie.voltagem,
    ie.potencia_watts,
    ie.cor,
    im.preco_custo
FROM invent_eletronicos ie
JOIN invent_mestre im ON ie.id_item = im.id_item
ORDER BY im.preco_custo DESC
'''
display(pd.read_sql_query(q5, conn))


,codigo_sku,descricao,marca,garantia_meses,voltagem,potencia_watts,cor,preco_custo
0,SKU-ELE-001,Notebook Pro 15,TechPro,12,Bivolt,65,Prata,3500
1,SKU-ELE-002,"Monitor UltraWide 29""",VisionMax,24,Bivolt,45,Preto,1200
2,SKU-ELE-010,Impressora Multifuncional,PrintAll,12,Bivolt,30,Branco,900
3,SKU-ELE-005,Fone Bluetooth ANC,SoundWave,12,None,2,Branco,600
4,SKU-ELE-007,SSD NVMe 1TB,DataFast,60,None,4,None,450
5,SKU-ELE-008,Roteador Wi-Fi 6,NetSpeed,24,Bivolt,12,Preto,380
6,SKU-ELE-003,Teclado Mecânico RGB,KeyForce,12,Bivolt,5,Preto,250
7,SKU-ELE-006,Webcam Full HD,ClearView,6,Bivolt,3,Preto,220
8,SKU-ELE-004,Mouse Gamer Óptico,KeyForce,12,Bivolt,3,Preto,150
9,SKU-ELE-009,Carregador USB-C 65W,PowerGo,6,Bivolt,65,Branco,120


## Cassandra — modelagem orientada a análise histórica

No Cassandra não modelamos por normalização (como em SQL) e sim **por consulta**: cada pergunta analítica vira uma tabela própria, com a chave de partição escolhida para agrupar exatamente os dados daquela consulta e as clustering columns definindo a ordenação.

### Perguntas analíticas escolhidas (orientadas a histórico)

1. **Qual o histórico completo de movimentações de um item específico, em ordem cronológica?** (ex.: auditoria de um SKU ao longo do tempo)
2. **Qual o volume movimentado por departamento, mês a mês, ao longo do tempo?** (ex.: tendência de consumo/venda por período)
3. **Quais foram as últimas movimentações de saída de um departamento, da mais recente para a mais antiga?** (ex.: painel operacional "o que saiu por último")

### Modelagem das tabelas (CQL)

* `historico_por_item` — partição por `id_item`, clustering por `data_transacao, id_transacao`
* `volume_mensal_departamento` — partição composta por `(id_departamento, ano_mes)`, clustering por `data_transacao`
* `ultimas_saidas_departamento` — partição composta por `(id_departamento, tipo)`, clustering por `data_transacao DESC, id_transacao DESC`


In [13]:
!pip install -q cassandra-driver

### Como conectar - passo a passo

Decidímos testar o funcionamento do notebook no ambiente local dentro do WSL.

**Cassandra local via Docker**:
```
docker run --name cassandra-lab -p 9042:9042 -d cassandra:4.1
```
Aguardar ~1 minuto para o nó subir e usar `Cluster(['127.0.0.1'])` em vez do bloco de conexão com Astra


In [22]:
from cassandra.cluster import Cluster

try:
    cluster = Cluster(['127.0.0.1'], port=9042)
    session = cluster.connect()
    print("Conectado ao Cassandra com sucesso!")

except Exception as e:
    print(f"Erro ao conectar ao Cassandra: {str(e)[:100]}")
    session = None


Conectado ao Cassandra com sucesso!


In [23]:
if session:
    KEYSPACE = "armazem_lab"

    session.execute(f'''
        CREATE KEYSPACE IF NOT EXISTS {KEYSPACE}
        WITH replication = {{'class': 'SimpleStrategy', 'replication_factor': 1}}
    ''')

    session.set_keyspace(KEYSPACE)

    session.execute('''
    CREATE TABLE IF NOT EXISTS historico_por_item (
        id_item int,
        data_transacao date,
        id_transacao int,
        tipo text,
        quantidade int,
        origem_destino text,
        PRIMARY KEY (id_item, data_transacao, id_transacao)
    ) WITH CLUSTERING ORDER BY (data_transacao ASC, id_transacao ASC)
    ''')

    session.execute('''
    CREATE TABLE IF NOT EXISTS volume_mensal_departamento (
        id_departamento int,
        ano_mes text,
        data_transacao date,
        id_item int,
        id_transacao int,
        tipo text,
        quantidade int,
        PRIMARY KEY ((id_departamento, ano_mes), data_transacao, id_item, id_transacao)
    ) WITH CLUSTERING ORDER BY (data_transacao DESC, id_item ASC, id_transacao ASC)
    ''')

    session.execute('''
    CREATE TABLE IF NOT EXISTS ultimas_saidas_departamento (
        id_departamento int,
        tipo text,
        data_transacao date,
        id_transacao int,
        id_item int,
        quantidade int,
        PRIMARY KEY ((id_departamento, tipo), data_transacao, id_transacao)
    ) WITH CLUSTERING ORDER BY (data_transacao DESC, id_transacao DESC)
    ''')

    print("Keyspace e tabelas prontos no Cassandra.")

else:
    print("Cassandra não disponível. Pulando criação de tabelas.")

Keyspace e tabelas prontos no Cassandra.


### Gerando e inserindo dados

Reaproveitamos as mesmas 22 movimentações já usadas no SQL (mesmo domínio, mesmas datas relativas), para que os dois modelos — relacional e Cassandra — representem o mesmo histórico e os resultados sejam comparáveis.


In [24]:
if session:
    ID_DEPARTAMENTO = 1

    hoje = datetime.now()
    def dias_atras(n):
        return hoje - timedelta(days=n)

    transacoes_cassandra = [
        (1, dias_atras(29), 1, 'ENTRADA', 5, 'Fornecedor A'),
        (2, dias_atras(28), 1, 'ENTRADA', 3, 'Fornecedor B'),
        (3, dias_atras(27), 2, 'ENTRADA', 2, 'Fornecedor C'),
        (4, dias_atras(26), 3, 'ENTRADA', 10, 'Fornecedor A'),
        (5, dias_atras(25), 4, 'ENTRADA', 15, 'Fornecedor D'),
        (6, dias_atras(24), 5, 'ENTRADA', 8, 'Fornecedor E'),
        (7, dias_atras(23), 6, 'ENTRADA', 4, 'Fornecedor F'),
        (8, dias_atras(22), 7, 'ENTRADA', 12, 'Fornecedor G'),
        (9, dias_atras(21), 8, 'ENTRADA', 20, 'Fornecedor H'),
        (10, dias_atras(20), 9, 'ENTRADA', 6, 'Fornecedor I'),
        (11, dias_atras(19), 1, 'SAIDA', -2, 'Departamento Vendas'),
        (12, dias_atras(18), 2, 'SAIDA', -1, 'Departamento RH'),
        (13, dias_atras(17), 3, 'SAIDA', -3, 'Departamento TI'),
        (14, dias_atras(16), 4, 'SAIDA', -5, 'Departamento Financeiro'),
        (15, dias_atras(15), 5, 'SAIDA', -2, 'Departamento Vendas'),
        (16, dias_atras(14), 6, 'SAIDA', -1, 'Departamento RH'),
        (17, dias_atras(13), 7, 'SAIDA', -4, 'Departamento TI'),
        (18, dias_atras(12), 8, 'SAIDA', -6, 'Departamento Financeiro'),
        (19, dias_atras(11), 9, 'SAIDA', -2, 'Departamento Vendas'),
        (20, dias_atras(11), 10, 'SAIDA', -3, 'Departamento TI'),
        (21, dias_atras(11), 3, 'AJUSTE', -2, 'Inventário físico - quebra'),
        (22, dias_atras(10), 6, 'AJUSTE', 3,  'Inventário físico - sobra'),
    ]

    ins_historico = session.prepare('''
        INSERT INTO historico_por_item (id_item, data_transacao, id_transacao, tipo, quantidade, origem_destino)
        VALUES (?, ?, ?, ?, ?, ?)
    ''')

    for transacao in transacoes_cassandra:
        session.execute(ins_historico, transacao)

    print(f"{len(transacoes_cassandra)} movimentações inseridas nas tabelas do Cassandra.")
else:
    print("Cassandra não disponível. Pulando inserção de dados.")


22 movimentações inseridas nas tabelas do Cassandra.


### Testando as queries

**1. Histórico cronológico de movimentações de um item específico (ex.: item 1 — Notebook Pro 15):**

In [25]:
if session:
    rows = session.execute("SELECT * FROM historico_por_item WHERE id_item = 1")
    display(pd.DataFrame(list(rows)))

else:
    print("Cassandra não disponível.")

,id_item,data_transacao,id_transacao,origem_destino,quantidade,tipo
0,1,2026-07-20,1,Fornecedor A,5,ENTRADA


**2. Volume movimentado por departamento, mês a mês:**

In [ ]:
if session:
    ano_mes_atual = datetime.now().strftime('%Y-%m')

    ins_volume = session.prepare('''
        INSERT INTO volume_mensal_departamento (id_departamento, ano_mes, data_transacao, id_item, id_transacao, tipo, quantidade)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''')

    volume_data = [
        (ID_DEPARTAMENTO, ano_mes_atual, dias_atras(5), 1, 1, 'ENTRADA', 5),
        (ID_DEPARTAMENTO, ano_mes_atual, dias_atras(3), 2, 2, 'ENTRADA', 3),
        (ID_DEPARTAMENTO, ano_mes_atual, dias_atras(2), 1, 3, 'SAIDA', -2),
    ]

    for data in volume_data:
        session.execute(ins_volume, data)

    rows = session.execute(
        "SELECT * FROM volume_mensal_departamento WHERE id_departamento = %s AND ano_mes = %s",
        (ID_DEPARTAMENTO, ano_mes_atual)
    )
    df_volume = pd.DataFrame(list(rows))
    display(df_volume)
    if not df_volume.empty:
        print("Volume total no mês corrente:", df_volume['quantidade'].abs().sum())

else:
    print("Cassandra não disponível.")


,id_departamento,ano_mes,data_transacao,id_item,id_transacao,quantidade,tipo
0,1,2026-08,2026-08-16,1,3,-2,SAIDA
1,1,2026-08,2026-08-15,2,2,3,ENTRADA
2,1,2026-08,2026-08-13,1,1,5,ENTRADA


Volume total no mês corrente: 10


**3. Últimas movimentações de saída do departamento

In [ ]:
if session:
    ins_saidas = session.prepare('''
        INSERT INTO ultimas_saidas_departamento (id_departamento, tipo, data_transacao, id_transacao, id_item, quantidade)
        VALUES (?, ?, ?, ?, ?, ?)
    ''')

    saidas_data = [
        (ID_DEPARTAMENTO, 'SAIDA', dias_atras(2), 1, 1, -2),
        (ID_DEPARTAMENTO, 'SAIDA', dias_atras(1), 2, 2, -1),
        (ID_DEPARTAMENTO, 'SAIDA', dias_atras(0), 3, 3, -3),
    ]

    for data in saidas_data:
        session.execute(ins_saidas, data)

    rows = session.execute(
        "SELECT * FROM ultimas_saidas_departamento WHERE id_departamento = %s AND tipo = 'SAIDA' LIMIT 5",
        (ID_DEPARTAMENTO,)
    )
    display(pd.DataFrame(list(rows)))

else:
    print("Cassandra não disponível.")


,id_departamento,tipo,data_transacao,id_transacao,id_item,quantidade
0,1,SAIDA,2026-08-18,3,3,-3
1,1,SAIDA,2026-08-17,2,2,-1
2,1,SAIDA,2026-08-16,1,1,-2
